# 01 - NCRB 2024 Curated Data Inventory & Pipeline Verification

This notebook serves as the initial exploratory data inventory and verification gateway for the **Big Data Analytics (BDA) Project** on the **NCRB 2024** crime dataset.

### Dual-Tier Architecture
- **Tier 1 (District Core - 9 Files)**: All-India geographic coverage across ~1,000+ administrative units for spatial clustering, crime severity indices, and vulnerable group profiling.
- **Tier 2 (Metropolitan Analytical - 8 Files)**: Multi-year trends (2022–2024), causal motive decomposition (murder, cybercrime), and socio-demographic determinants of delinquency across India's million-plus cities.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Project paths
PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT.resolve()))

from src.curated_loader import load_manifest, verify_portfolio, clean_district_table, clean_metro_table

print("Project Root:", PROJECT_ROOT.resolve())
print("Curated data modules imported successfully!")

Project Root: C:\Users\ayush\OneDrive\Desktop\Ayush\BDA Project
Curated data modules imported successfully!


## 1. Curated Portfolio Manifest Inspection
We load the formal portfolio catalog registered in `data/curated_portfolio_manifest.csv`.

In [2]:
manifest_csv = PROJECT_ROOT / "data" / "curated_portfolio_manifest.csv"
df_manifest = pd.read_csv(manifest_csv)

print(f"Total Curated Datasets: {len(df_manifest)}")
display(df_manifest[["file_id", "tier", "table_code", "domain", "temporal_coverage"]])

Total Curated Datasets: 17


,file_id,tier,table_code,domain,temporal_coverage
0,DIST_01_IPC,Tier 1: District Core,District-IPC-2024,Overall IPC / BNS Crimes,2024
1,DIST_02_SLL,Tier 1: District Core,District-SLL-2024,Special & Local Laws (SLL),2024
2,DIST_03_WOMEN,Tier 1: District Core,District-Women-2024,Vulnerable Groups: Women,2024
3,DIST_04_CHILDREN,Tier 1: District Core,District-Children-2024,Vulnerable Groups: Children,2024
4,DIST_05_SC,Tier 1: District Core,District-SC-2024,Marginalized Demographics: SCs,2024
5,DIST_06_ST,Tier 1: District Core,District-ST-2024,Marginalized Demographics: STs,2024
6,DIST_07_JUVENILE_IPC,Tier 1: District Core,District-JuvenileIPC-2024,Youth Delinquency: IPC/BNS,2024
7,DIST_09_CYBER,Tier 1: District Core,District-Cyber-2024,Digital / Tech-Enabled Crime,2024
8,DIST_10_MISSING,Tier 1: District Core,District-Missing-2024,Social Vulnerability: Missing Persons,2024
9,METRO_01_TREND,Tier 2: Metropolitan Analytical,TABLE 1B.1,Metropolitan Macro Trends & Policing Efficiency,2022-2024


## 2. Automated File Health & Dimension Verification
Verify that every curated file is present on disk, readable, and report its raw dimensions.

In [3]:
df_health = verify_portfolio()
display(df_health[["File ID", "Tier", "File Name", "Size (KB)", "Raw Rows", "Raw Cols", "Status"]])
assert (df_health["Status"] == "OK").all(), "All datasets must have Status OK!"

,File ID,Tier,File Name,Size (KB),Raw Rows,Raw Cols,Status
0,DIST_01_IPC,Tier 1: District Core,1DistrictwiseIPCCrimes2024.xlsx,718.5,1087,156,OK
1,DIST_02_SLL,Tier 1: District Core,2DistrictwiseSLLCrimes2024.xlsx,433.0,1086,95,OK
2,DIST_03_WOMEN,Tier 1: District Core,3DistrictwiseCrimeagainstWomen2024.xlsx,324.3,1087,67,OK
3,DIST_04_CHILDREN,Tier 1: District Core,4DistrictwiseCrimeagainstChildren2024.xlsx,330.9,1087,76,OK
4,DIST_05_SC,Tier 1: District Core,5DistrictwiseCrimeagainstSCs2024.xlsx,229.9,1079,53,OK
5,DIST_06_ST,Tier 1: District Core,6DistrictwiseCrimeagainstSTs2024.xlsx,218.3,1086,53,OK
6,DIST_07_JUVENILE_IPC,Tier 1: District Core,7DistrictwiseIPCCrimebyJuveniles2024.xlsx,551.4,1087,156,OK
7,DIST_09_CYBER,Tier 1: District Core,9DistrictwiseCyberCrimes2024.xlsx,262.5,1087,60,OK
8,DIST_10_MISSING,Tier 1: District Core,10DistrictwiseMissingPersons2024.xlsx,150.7,1084,26,OK
9,METRO_01_TREND,Tier 2: Metropolitan Analytical,TABLE1B16.xlsx,12.2,44,8,OK


## 3. Tier 1: District Core Ingestion Deep Dive
We test the district parser on `1DistrictwiseIPCCrimes2024.xlsx` to extract clean state/district observations without summary headers.

In [4]:
dist_ipc_path = PROJECT_ROOT / "data" / "curated" / "district" / "1DistrictwiseIPCCrimes2024.xlsx"
df_dist_ipc = clean_district_table(dist_ipc_path)

print(f"Cleaned District DataFrame Shape: {df_dist_ipc.shape}")
print(f"Total Administrative Units: {len(df_dist_ipc)}")
print(f"Total States/UTs Covered: {df_dist_ipc['State'].nunique()}")
display(df_dist_ipc.head())

Cleaned District DataFrame Shape: (965, 155)
Total Administrative Units: 965
Total States/UTs Covered: 37


,State,District,Offences against Women and Child - Rape (Section 64 to Section 71 BNS / Section 376 IPC),"Attempt to Commit Rape (Section 64-68,70-71 r/w Sec 62 BNS Sec 376 r/w 511 IPC)",Sexual Intercourse by employing deceitful means (Section 69 BNS),Disclosure of Identity of Victim( Section 72(1) BNS),Assault or use of Criminal Force to woman with intent to outrage her modesty (Section 74 BNS/Sec 354 IPC),Sexual Harassment (Section 75 BNS/ Section 354A IPC) - Sexual Harassment (Total a+b+c+d+e+f),a) At Office Premises,b) At Work Premises (Shops/Malls/Factories/Home etc.),...,Obscene acts and songs at public Places (Section 296 BNS/Section 294 IPC),Offences relating to Religion (Section 298 to 301 BNS/Section 295-297 IPC),Cheating by Impersonation (Section 319 BNS/Section 419 IPC)),"Offences related to Mischief(324(5)(6),326(a)(b)(c)(d) BNS/Section 428-433 IPC)","Arson (Section 326(f),326(G) & 327 (2) BNS/Section 435,436 & 438 IPC)",Criminal Trespass (Section 329(3) BNS/Section 447-252 IPC),Circulate False/Fake News/Rumours (Section 353 BNS/ Section 353 r/w IT Act/Section 505 IPC r/w IT Act),Miscellaneous IPC/BNS Crimes (Total),Other IPC/BNS crimes,Total Cognizable IPC/BNS crimes
0,UNKNOWN,[2],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Andhra Pradesh,Alluri Sitharama Raju,18,0,3,0,5,11,2,2,...,0,0,1,1,1,22,2,112,114,795
2,Andhra Pradesh,Anakapalli,15,2,0,0,76,29,0,0,...,6,0,3,15,19,157,8,677,128,2979
3,Andhra Pradesh,Anantapuramu,11,4,1,0,86,29,0,0,...,2,2,2,3,40,137,5,1082,695,5194
4,Andhra Pradesh,Annamayya,5,0,0,0,14,16,0,0,...,0,1,0,67,5,101,0,957,465,3755


## 4. Tier 2: Metropolitan Comparative Layer Deep Dive
We test the metropolitan parser on `TABLE1B16.xlsx` (Table 1B.1) covering 3-year IPC crime trends (2022-2024) across metropolitan cities.

In [5]:
metro_trend_path = PROJECT_ROOT / "data" / "curated" / "metropolitan" / "TABLE1B16.xlsx"
df_metro_trend = clean_metro_table(metro_trend_path)

print(f"Cleaned Metropolitan DataFrame Shape: {df_metro_trend.shape}")
print(f"Total Metropolitan Cities: {len(df_metro_trend)}")
display(df_metro_trend.head())

Cleaned Metropolitan DataFrame Shape: (38, 7)
Total Metropolitan Cities: 38


,City,2022,2023,2024,Actual Population (in Lakhs) (2011)+,Rate of Cognizable Crimes (IPC) (2024),Chargesheeting Rate (2024)
0,[2],[3],[4],[5],[6],[7],[8]
1,Agra,6947,7026,8066,17.5,462,80.9
2,Amritsar,2815,2990,2245,11.8,189.6,70.6
3,Asansol,4519,4568,4702,12.4,378.3,83.9
4,Chhatrapati Sambhajinagar,5345,5814,5994,11.9,504.1,68


## 5. Next Steps
1. **Batch Pipeline Ingestion**: Execute feature extraction scripts across all 9 district files to build the unified wide feature matrix.
2. **Geospatial & Spatial Clustering**: Apply K-Means and DBSCAN on standardized crime density features.
3. **Metropolitan Quadrant Analysis**: Map cities on Crime Rate vs. Chargesheeting Rate efficiency axes.